# Account Site A/B Test Analysis

**Experiment ID:** `019be2ce`

This notebook analyzes the Shoplift A/B test on the Account site. We examine user behavior, subscription cancellations, and new orders across the two test branches.

---

In [ ]:
# Experiment configuration variables
EXPERIMENT_ID = '019be2ce'
VARIANT_ID = '019be2a5-1302-7f60-924d-9505759ad501'
CONTROL_ID = '019be2a4-a17f-782e-bdbd-5a2dbad306d2'
TEST_START_DATE = '2026-01-01'
TEST_END_DATE = '2026-06-12'

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Control Variant ID: {CONTROL_ID}")
print(f"Variant (Treatment) ID: {VARIANT_ID}")
print(f"Test Start Date: {TEST_START_DATE}")
print(f"Test End Date: {TEST_END_DATE}")

In [ ]:
%%sql -r setup_context
USE DATABASE ANALYTICS;
USE SCHEMA GA4;

## Step 1: Identify Test Population & Remove Internal Users

We filter the GA4 events for the experiment `019be2ce` and exclude internal users (employees/testers) based on known internal email domains or tags.

In [ ]:
%%sql -r step1_overview
-- Base test population: all events from the experiment, excluding internal users
CREATE OR REPLACE TEMPORARY TABLE _test_events AS
WITH experiment_events AS (
    SELECT
        e.*,
        CASE
            WHEN e.shoplift_variant_ids LIKE '%{{VARIANT_ID}}%' THEN 'Variant'
            WHEN e.shoplift_variant_ids LIKE '%{{CONTROL_ID}}%' THEN 'Control'
        END AS test_branch
    FROM ANALYTICS.GA4.FCT_GA4__WEBSITE_EVENTS e
    WHERE e.shoplift_experiment_ids LIKE '%{{EXPERIMENT_ID}}%'
      AND e.event_date_utc >= '{{TEST_START_DATE}}'::DATE
      AND e.event_date_utc <= '{{TEST_END_DATE}}'::DATE
)
SELECT ee.*
FROM experiment_events ee
LEFT JOIN ANALYTICS.SHOPIFY.DIM_SHOPIFY__CUSTOMERS c
    ON ee.shopify_customer_id = c.customer_id
WHERE ee.test_branch IS NOT NULL
  -- Exclude internal users by email domain patterns
  AND COALESCE(c.email, ee.email, '') NOT LIKE '%@oneskin.co'
  AND COALESCE(c.email, ee.email, '') NOT LIKE '%@shoplift.ai'
  AND LOWER(COALESCE(c.tags, '')) NOT LIKE '%internal%'
  AND LOWER(COALESCE(c.tags, '')) NOT LIKE '%employee%';

SELECT COUNT(*) AS total_events, 
       COUNT(DISTINCT canonical_id) AS total_users,
       COUNT(DISTINCT session_id) AS total_sessions,
       MIN(event_date_utc) AS test_start_date,
       MAX(event_date_utc) AS test_end_date
FROM _test_events;

---
## Step 2: Number of Users and Sessions on Each Branch

Count of unique users (canonical_id) assigned to each test branch.

In [ ]:
%%sql -r step2_users
-- Step 2: Users per branch
SELECT
    test_branch,
    COUNT(DISTINCT canonical_id) AS unique_users,
    COUNT(DISTINCT session_id) AS unique_sessions,
    ROUND(COUNT(DISTINCT canonical_id) / SUM(COUNT(DISTINCT canonical_id)) OVER(), 3) * 100 as percentage_of_users
FROM _test_events
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
import matplotlib.pyplot as plt
print("--- Step 2: Users and Sessions per Test Branch ---")
print(step2_users.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(step2_users['TEST_BRANCH'], step2_users['UNIQUE_USERS'], color=['#2196F3', '#FF9800'])
axes[0].set_title('Unique Users per Test Branch')
axes[0].set_ylabel('Users')
axes[0].set_xlabel('Branch')
for i, v in enumerate(step2_users['UNIQUE_USERS']):
    axes[0].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[1].bar(step2_users['TEST_BRANCH'], step2_users['UNIQUE_SESSIONS'], color=['#2196F3', '#FF9800'])
axes[1].set_title('Unique Sessions per Test Branch')
axes[1].set_ylabel('Sessions')
axes[1].set_xlabel('Branch')
for i, v in enumerate(step2_users['UNIQUE_SESSIONS']):
    axes[1].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[2].bar(step2_users['TEST_BRANCH'], step2_users['PERCENTAGE_OF_USERS'], color=['#2196F3', '#FF9800'])
axes[2].set_title('% of Users per Test Branch')
axes[2].set_ylabel('% of Users')
axes[2].set_xlabel('Branch')
for i, v in enumerate(step2_users['PERCENTAGE_OF_USERS']):
    axes[2].text(i, v + v*0.01, f'{v:.1f}%', ha='center', fontweight='bold')
    
plt.tight_layout()
plt.show()

---
## Step 3: Users & Sessions Filtered by Account Site Visitors

Same metrics as Steps 2 & 3, but only for users/sessions that visited the Account site (page_location containing '/account').

In [ ]:
%%sql -r step4_account
-- Step 3: Users and sessions that visited the Account site
SELECT
    test_branch,
    COUNT(DISTINCT canonical_id) AS account_users,
    COUNT(DISTINCT session_id) AS account_sessions,
    ROUND(COUNT(DISTINCT canonical_id) / SUM(COUNT(DISTINCT canonical_id)) OVER(), 3) * 100 as percentage_of_users
FROM _test_events
WHERE page_type = 'Account'
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
print("--- Step 4: Account Site Visitors per Branch ---")
print(step4_account.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(step4_account['TEST_BRANCH'], step4_account['ACCOUNT_USERS'], color=['#2196F3', '#FF9800'])
axes[0].set_title('Account Site Users per Branch')
axes[0].set_ylabel('Users')
for i, v in enumerate(step4_account['ACCOUNT_USERS']):
    axes[0].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[1].bar(step4_account['TEST_BRANCH'], step4_account['ACCOUNT_SESSIONS'], color=['#2196F3', '#FF9800'])
axes[1].set_title('Account Site Sessions per Branch')
axes[1].set_ylabel('Sessions')
for i, v in enumerate(step4_account['ACCOUNT_SESSIONS']):
    axes[1].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[2].bar(step4_account['TEST_BRANCH'], step4_account['PERCENTAGE_OF_USERS'], color=['#2196F3', '#FF9800'])
axes[2].set_title('% Account Site Users per Branch')
axes[2].set_ylabel('% Users')
for i, v in enumerate(step4_account['PERCENTAGE_OF_USERS']):
    axes[2].text(i, v + v*0.01, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Step 4: Recognized Users (with Shopify Customer ID) visited the Account site

Filtering for users that have a `shopify_customer_id` populated - these are recognized/logged-in users and visited the Account site.

In [ ]:
%%sql -r step5_recognized
-- Step 4: Recognized users (with shopify_customer_id) per branch
SELECT
    test_branch,
    COUNT(DISTINCT canonical_id) AS recognized_users,
    COUNT(DISTINCT session_id) AS recognized_sessions,
    ROUND(COUNT(DISTINCT canonical_id) / SUM(COUNT(DISTINCT canonical_id)) OVER(), 3) * 100 as percentage_of_users
FROM _test_events
WHERE shopify_customer_id IS NOT NULL AND page_type = 'Account'
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
print("--- Step 5: Recognized Users (with Shopify Customer ID) per Branch ---")
print(step5_recognized.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(step5_recognized['TEST_BRANCH'], step5_recognized['RECOGNIZED_USERS'], color=['#2196F3', '#FF9800'])
axes[0].set_title('Recognized Users per Branch')
axes[0].set_ylabel('Users')
for i, v in enumerate(step5_recognized['RECOGNIZED_USERS']):
    axes[0].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[1].bar(step5_recognized['TEST_BRANCH'], step5_recognized['RECOGNIZED_SESSIONS'], color=['#2196F3', '#FF9800'])
axes[1].set_title('Recognized Sessions per Branch')
axes[1].set_ylabel('Sessions')
for i, v in enumerate(step5_recognized['RECOGNIZED_SESSIONS']):
    axes[1].text(i, v + v*0.01, f'{v:,}', ha='center', fontweight='bold')

axes[2].bar(step5_recognized['TEST_BRANCH'], step5_recognized['PERCENTAGE_OF_USERS'], color=['#2196F3', '#FF9800'])
axes[2].set_title('% Account Site Users per Branch')
axes[2].set_ylabel('% Users')
for i, v in enumerate(step5_recognized['PERCENTAGE_OF_USERS']):
    axes[2].text(i, v + v*0.01, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Step 5: Subscription Cancellation Rates (Same Day, 7/15/30 Days)

For users who visited the account site, calculate the percentage that cancelled a subscription (cancelled full subscription or removed a product) within various time windows after their first account visit.

In [ ]:
%%sql -r step6_timestamps
-- Step 5: First timestamp each user was seen in the test and entered the account site
CREATE OR REPLACE TEMPORARY TABLE _user_first_account_visit AS
SELECT
    shopify_customer_id,
    canonical_id,
    test_branch,
    MIN(event_timestamp_utc) AS first_test_timestamp,
    MIN(CASE WHEN page_type = 'Account' THEN event_timestamp_utc END) AS first_account_visit
FROM _test_events
WHERE shopify_customer_id IS NOT NULL
GROUP BY shopify_customer_id, canonical_id, test_branch
HAVING first_account_visit IS NOT NULL;

SELECT
    test_branch,
    COUNT(*) AS users_with_account_visit,
    DATE(MIN(first_test_timestamp)) AS earliest_test_entry,
    DATE(MIN(first_test_timestamp)) AS latest_test_entry,
    DATE(MIN(first_account_visit)) as first_account_visit,
    DATE(MAX(first_account_visit)) AS latest_account_visit
FROM _user_first_account_visit
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
%%sql -r step7_cancellations
-- Step 6: Cancellation rates within time windows
CREATE OR REPLACE TEMPORARY TABLE _cancellation_analysis AS
WITH user_cancellations AS (
    -- Full subscription cancellations
    SELECT
        u.shopify_customer_id,
        u.test_branch,
        u.first_account_visit,
        s.cancelled_at AS cancellation_timestamp
    FROM _user_first_account_visit u
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON u.shopify_customer_id = s.shopify_customer_id
    WHERE s.is_cancelled = TRUE
      AND s.cancelled_at >= u.first_account_visit
      AND u.first_account_visit IS NOT NULL

    UNION ALL

    -- Subscription line cancellations (individual product removals)
    SELECT
        u.shopify_customer_id,
        u.test_branch,
        u.first_account_visit,
        sl.removed_at AS cancellation_line_timestamp
    FROM _user_first_account_visit u
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON u.shopify_customer_id = s.shopify_customer_id
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTION_LINES sl
        ON s.subscription_id = sl.subscription_id
    WHERE sl.removed_at IS NOT NULL
      AND sl.removed_at >= u.first_account_visit
      AND u.first_account_visit IS NOT NULL
)
SELECT
    u.shopify_customer_id,
    u.test_branch,
    u.first_account_visit,
    MIN(uc.cancellation_timestamp) AS first_cancellation_at,
    CASE WHEN MIN(uc.cancellation_timestamp) IS NOT NULL THEN 1 ELSE 0 END AS has_cancellation,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uc.cancellation_timestamp)) = 0 THEN 1 ELSE 0 END AS cancelled_same_day,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uc.cancellation_timestamp)) <= 7 THEN 1 ELSE 0 END AS cancelled_within_7d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uc.cancellation_timestamp)) <= 15 THEN 1 ELSE 0 END AS cancelled_within_15d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uc.cancellation_timestamp)) <= 30 THEN 1 ELSE 0 END AS cancelled_within_30d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uc.cancellation_timestamp)) <= 90 THEN 1 ELSE 0 END AS cancelled_within_90d
FROM _user_first_account_visit u
LEFT JOIN user_cancellations uc
    ON u.shopify_customer_id = uc.shopify_customer_id
    AND u.test_branch = uc.test_branch
GROUP BY u.shopify_customer_id, u.test_branch, u.first_account_visit;

SELECT
    test_branch,
    COUNT(*) AS total_users,
    SUM(cancelled_same_day) AS cancelled_same_day,
    ROUND(100.0 * SUM(cancelled_same_day) / COUNT(*), 2) AS pct_cancelled_same_day,
    SUM(cancelled_within_7d) AS cancelled_within_7d,
    ROUND(100.0 * SUM(cancelled_within_7d) / COUNT(*), 2) AS pct_cancelled_7d,
    SUM(cancelled_within_15d) AS cancelled_within_15d,
    ROUND(100.0 * SUM(cancelled_within_15d) / COUNT(*), 2) AS pct_cancelled_15d,
    SUM(cancelled_within_30d) AS cancelled_within_30d,
    ROUND(100.0 * SUM(cancelled_within_30d) / COUNT(*), 2) AS pct_cancelled_30d,
    SUM(cancelled_within_90d) AS cancelled_within_90d,
    ROUND(100.0 * SUM(cancelled_within_90d) / COUNT(*), 2) AS pct_cancelled_90d
FROM _cancellation_analysis
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
import numpy as np 

print("--- Step 7: Subscription Cancellation Rates by Branch ---")
print(step7_cancellations.to_string(index=False))

# Visualization
metrics = ['PCT_CANCELLED_SAME_DAY', 'PCT_CANCELLED_7D', 'PCT_CANCELLED_15D', 'PCT_CANCELLED_30D', 'PCT_CANCELLED_90D']
labels = ['Same Day', 'Within 7 Days', 'Within 15 Days', 'Within 30 Days', 'Within 90 Days']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
width = 0.35

control = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Control'][metrics].values.flatten()
variant = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Variant'][metrics].values.flatten()

bars1 = ax.bar(x - width/2, control, width, label='Control', color='#2196F3')
bars2 = ax.bar(x + width/2, variant, width, label='Variant', color='#FF9800')

ax.set_ylabel('Cancellation Rate (%)')
ax.set_title('Subscription Cancellation Rates by Time Window')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.bar_label(bars1, fmt='%.1f%%', padding=3)
ax.bar_label(bars2, fmt='%.1f%%', padding=3)
plt.tight_layout()
plt.show()

---
## Step 6: Compare Cancellation Rates vs Historical Baseline

Compare test branch cancellation rates to the monthly historical average cancellation rate from the 6 months before the test started.

In [ ]:
%%sql -r step8_baseline
-- Step 6: Historical baseline cancellation rate (6 months before test)
-- For every shopify_customer_id with events in the 6 months before the test,
-- find their first account visit, then compute cancellation rate (subscription + subscription_line)
-- for same day / 15 / 30 / 90 day windows after that account visit.
WITH historical_account_visitors AS (
    SELECT
        e.shopify_customer_id::TEXT AS shopify_customer_id,
        MIN(CASE WHEN page_type = 'Account' THEN e.event_timestamp_utc END) AS first_account_visit
    FROM ANALYTICS.GA4.FCT_GA4__WEBSITE_EVENTS e
    WHERE e.event_timestamp_utc >= DATEADD('month', -6, '{{TEST_START_DATE}}'::TIMESTAMP)
      AND e.event_timestamp_utc < '{{TEST_START_DATE}}'::TIMESTAMP
      AND e.shopify_customer_id IS NOT NULL
    GROUP BY e.shopify_customer_id::TEXT
    HAVING first_account_visit IS NOT NULL
),
user_cancellations AS (
    -- Full subscription cancellations
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        s.cancelled_at AS cancellation_timestamp
    FROM historical_account_visitors hav
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON hav.shopify_customer_id = s.shopify_customer_id
    WHERE s.is_cancelled = TRUE
      AND s.cancelled_at >= hav.first_account_visit

    UNION ALL

    -- Subscription line removals
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        sl.removed_at AS cancellation_timestamp
    FROM historical_account_visitors hav
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON hav.shopify_customer_id = s.shopify_customer_id
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTION_LINES sl
        ON s.subscription_id = sl.subscription_id
    WHERE sl.removed_at IS NOT NULL
      AND sl.removed_at >= hav.first_account_visit
),
baseline_analysis AS (
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        MIN(uc.cancellation_timestamp) AS first_cancellation_at
    FROM historical_account_visitors hav
    LEFT JOIN user_cancellations uc
        ON hav.shopify_customer_id = uc.shopify_customer_id
    GROUP BY hav.shopify_customer_id, hav.first_account_visit
)
SELECT
    'Historical Baseline' AS cohort,
    COUNT(*) AS total_users,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) = 0 THEN 1 ELSE 0 END) AS cancelled_same_day,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cancelled_same_day,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 7 THEN 1 ELSE 0 END) AS cancelled_within_7d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 7 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cancelled_7d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 15 THEN 1 ELSE 0 END) AS cancelled_within_15d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 15 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cancelled_15d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 30 THEN 1 ELSE 0 END) AS cancelled_within_30d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 30 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cancelled_30d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 90 THEN 1 ELSE 0 END) AS cancelled_within_90d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_cancellation_at) <= 90 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cancelled_90d
FROM baseline_analysis;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("--- Step 6: Historical Baseline vs Test Branches ---")
print("\nHistorical Baseline (6 months pre-test, account page visitors):")
print(step8_baseline.to_string(index=False))

# Extract baseline values
baseline = step8_baseline.iloc[0]

# Extract test branch values
control = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Control'].iloc[0]
variant = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Variant'].iloc[0]

print(f"\nComparison Table:")
comparison = pd.DataFrame({
    'Window': ['Same Day', '7 Days', '15 Days', '30 Days', '90 Days'],
    'Historical Baseline': [
        f"{baseline['PCT_CANCELLED_SAME_DAY']:.2f}%",
        f"{baseline['PCT_CANCELLED_7D']:.2f}%",
        f"{baseline['PCT_CANCELLED_15D']:.2f}%",
        f"{baseline['PCT_CANCELLED_30D']:.2f}%",
        f"{baseline['PCT_CANCELLED_90D']:.2f}%"
    ],
    'Control': [
        f"{control['PCT_CANCELLED_SAME_DAY']:.2f}%",
        f"{control['PCT_CANCELLED_7D']:.2f}%",
        f"{control['PCT_CANCELLED_15D']:.2f}%",
        f"{control['PCT_CANCELLED_30D']:.2f}%",
        f"{control['PCT_CANCELLED_90D']:.2f}%"
    ],
    'Variant': [
        f"{variant['PCT_CANCELLED_SAME_DAY']:.2f}%",
        f"{variant['PCT_CANCELLED_7D']:.2f}%",
        f"{variant['PCT_CANCELLED_15D']:.2f}%",
        f"{variant['PCT_CANCELLED_30D']:.2f}%",
        f"{variant['PCT_CANCELLED_90D']:.2f}%"
    ]
})
print(comparison.to_string(index=False))

# Visualization
windows = ['Same Day', '7 Days', '15 Days', '30 Days', '90 Days']
baseline_vals = [baseline['PCT_CANCELLED_SAME_DAY'], baseline['PCT_CANCELLED_7D'], baseline['PCT_CANCELLED_15D'], baseline['PCT_CANCELLED_30D'], baseline['PCT_CANCELLED_90D']]
control_vals = [control['PCT_CANCELLED_SAME_DAY'], control['PCT_CANCELLED_7D'], control['PCT_CANCELLED_15D'], control['PCT_CANCELLED_30D'], control['PCT_CANCELLED_90D']]
variant_vals = [variant['PCT_CANCELLED_SAME_DAY'], variant['PCT_CANCELLED_7D'], variant['PCT_CANCELLED_15D'], variant['PCT_CANCELLED_30D'], variant['PCT_CANCELLED_90D']]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(windows))
width = 0.25

bars1 = ax.bar(x - width, baseline_vals, width, label='Historical Baseline', color='#9E9E9E')
bars2 = ax.bar(x, control_vals, width, label='Control', color='#2196F3')
bars3 = ax.bar(x + width, variant_vals, width, label='Variant', color='#FF9800')

ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=8)
ax.bar_label(bars3, fmt='%.1f%%', padding=3, fontsize=8)

ax.set_ylabel('Cancellation Rate (%)')
ax.set_title('Cancellation Rate by Time Window: Historical Baseline vs Test Branches')
ax.set_xticks(x)
ax.set_xticklabels(windows)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 7: New Order Rates (Excluding Recurring Subscriptions)

Repeat the cancellation analysis but for new orders. Including new orders OTP or Subscription orders created after the Account visit. 

It excludes recurring subscription orders and includes new subscription lines (products subscribed.)

In [ ]:
%%sql -r step9_orders
-- Step 7: New order rates within time windows
CREATE OR REPLACE TEMPORARY TABLE _new_order_analysis AS
WITH user_new_orders AS (
    -- Shopify OTP or Initial orders
    SELECT
        u.shopify_customer_id,
        u.test_branch,
        u.first_account_visit,
        o.order_created_at AS order_timestamp
    FROM _user_first_account_visit u
    INNER JOIN ANALYTICS.SHOPIFY.FCT_SHOPIFY__ORDERS o
        ON u.shopify_customer_id::NUMBER = o.customer_id
    WHERE (o.primary_order_type = 'OTP' OR o.initial_vs_recurring_flag = 'Initial')
      AND o.order_created_at >= u.first_account_visit
      AND o.is_cancelled = FALSE
      AND u.first_account_visit IS NOT NULL

    UNION ALL

    -- New Skio subscription created after account visit
    SELECT
        u.shopify_customer_id,
        u.test_branch,
        u.first_account_visit,
        s.created_at AS order_timestamp
    FROM _user_first_account_visit u
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON u.shopify_customer_id = s.shopify_customer_id
    WHERE s.created_at >= u.first_account_visit -- new subscription
      AND u.first_account_visit IS NOT NULL

    UNION ALL 
    
    -- New Skio subscription lines created after account visit
    SELECT
        u.shopify_customer_id,
        u.test_branch,
        u.first_account_visit,
        sl.created_at AS order_timestamp
    FROM _user_first_account_visit u
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON u.shopify_customer_id = s.shopify_customer_id
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTION_LINES sl
        ON s.subscription_id = sl.subscription_id
    WHERE sl.created_at >= u.first_account_visit -- new line added after initial sub creation
      AND u.first_account_visit IS NOT NULL
)
SELECT
    u.shopify_customer_id,
    u.test_branch,
    u.first_account_visit,
    MIN(uno.order_timestamp) AS first_new_order_at,
    CASE WHEN MIN(uno.order_timestamp) IS NOT NULL THEN 1 ELSE 0 END AS has_new_order,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uno.order_timestamp)) = 0 THEN 1 ELSE 0 END AS ordered_same_day,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uno.order_timestamp)) <= 7 THEN 1 ELSE 0 END AS ordered_within_7d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uno.order_timestamp)) <= 15 THEN 1 ELSE 0 END AS ordered_within_15d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uno.order_timestamp)) <= 30 THEN 1 ELSE 0 END AS ordered_within_30d,
    CASE WHEN DATEDIFF('day', u.first_account_visit, MIN(uno.order_timestamp)) <= 90 THEN 1 ELSE 0 END AS ordered_within_90d
FROM _user_first_account_visit u
LEFT JOIN user_new_orders uno
    ON u.shopify_customer_id = uno.shopify_customer_id
    AND u.test_branch = uno.test_branch
WHERE u.first_account_visit IS NOT NULL
GROUP BY u.shopify_customer_id, u.test_branch, u.first_account_visit;

SELECT
    test_branch,
    COUNT(*) AS total_users,
    SUM(ordered_same_day) AS ordered_same_day,
    ROUND(100.0 * SUM(ordered_same_day) / COUNT(*), 2) AS pct_ordered_same_day,
    SUM(ordered_within_7d) AS ordered_within_7d,
    ROUND(100.0 * SUM(ordered_within_7d) / COUNT(*), 2) AS pct_ordered_7d,
    SUM(ordered_within_15d) AS ordered_within_15d,
    ROUND(100.0 * SUM(ordered_within_15d) / COUNT(*), 2) AS pct_ordered_15d,
    SUM(ordered_within_30d) AS ordered_within_30d,
    ROUND(100.0 * SUM(ordered_within_30d) / COUNT(*), 2) AS pct_ordered_30d,
    SUM(ordered_within_90d) AS ordered_within_90d,
    ROUND(100.0 * SUM(ordered_within_90d) / COUNT(*), 2) AS pct_ordered_90d
FROM _new_order_analysis
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
print("--- Step 7: New Order Rates by Branch (Excluding Recurring) ---")
print(step9_orders.to_string(index=False))

metrics = ['PCT_ORDERED_SAME_DAY', 'PCT_ORDERED_7D', 'PCT_ORDERED_15D', 'PCT_ORDERED_30D', 'PCT_ORDERED_90D']
labels = ['Same Day', 'Within 7 Days', 'Within 15 Days', 'Within 30 Days', 'Within 90 Days']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
width = 0.35

control = step9_orders[step9_orders['TEST_BRANCH'] == 'Control'][metrics].values.flatten()
variant = step9_orders[step9_orders['TEST_BRANCH'] == 'Variant'][metrics].values.flatten()

bars1 = ax.bar(x - width/2, control, width, label='Control', color='#2196F3')
bars2 = ax.bar(x + width/2, variant, width, label='Variant', color='#FF9800')

ax.set_ylabel('New Order Rate (%)')
ax.set_title('New Order Rates by Time Window (Excl. Recurring Subscriptions)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.bar_label(bars1, fmt='%.1f%%', padding=3)
ax.bar_label(bars2, fmt='%.1f%%', padding=3)
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r step9_baseline
-- Step 8 continued: Historical baseline for new orders
-- Same logic as Step 8: for every shopify_customer_id with events in the 6 months before the test,
-- find their first account visit, then compute new order rate (OTP/Initial + new Skio subs/lines)
-- for same day / 7 / 15 / 30 / 90 day windows after that account visit.
WITH historical_account_visitors AS (
    SELECT
        e.shopify_customer_id::TEXT AS shopify_customer_id,
        MIN(CASE WHEN LOWER(e.page_location) LIKE '%/account%' THEN e.event_timestamp_utc END) AS first_account_visit
    FROM ANALYTICS.GA4.FCT_GA4__WEBSITE_EVENTS e
    WHERE e.event_timestamp_utc >= DATEADD('month', -6, '{{TEST_START_DATE}}'::TIMESTAMP)
      AND e.event_timestamp_utc < '{{TEST_START_DATE}}'::TIMESTAMP
      AND e.shopify_customer_id IS NOT NULL
    GROUP BY e.shopify_customer_id::TEXT
    HAVING first_account_visit IS NOT NULL
),
user_new_orders AS (
    -- Shopify OTP or Initial orders
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        o.order_created_at AS order_timestamp
    FROM historical_account_visitors hav
    INNER JOIN ANALYTICS.SHOPIFY.FCT_SHOPIFY__ORDERS o
        ON hav.shopify_customer_id = o.customer_id::TEXT
    WHERE (o.primary_order_type = 'OTP' OR o.initial_vs_recurring_flag = 'Initial')
      AND o.order_created_at >= hav.first_account_visit
      AND o.is_cancelled = FALSE

    UNION ALL

    -- New Skio subscriptions created after account visit
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        s.created_at AS order_timestamp
    FROM historical_account_visitors hav
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON hav.shopify_customer_id = s.shopify_customer_id
    WHERE s.created_at >= hav.first_account_visit

    UNION ALL

    -- New Skio subscription lines created after account visit
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        sl.created_at AS order_timestamp
    FROM historical_account_visitors hav
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTIONS s
        ON hav.shopify_customer_id = s.shopify_customer_id
    INNER JOIN ANALYTICS.SKIO.FCT_SKIO__SUBSCRIPTION_LINES sl
        ON s.subscription_id = sl.subscription_id
    WHERE sl.created_at >= hav.first_account_visit
),
baseline_analysis AS (
    SELECT
        hav.shopify_customer_id,
        hav.first_account_visit,
        MIN(uno.order_timestamp) AS first_new_order_at
    FROM historical_account_visitors hav
    LEFT JOIN user_new_orders uno
        ON hav.shopify_customer_id = uno.shopify_customer_id
    GROUP BY hav.shopify_customer_id, hav.first_account_visit
)
SELECT
    'Historical Baseline' AS cohort,
    COUNT(*) AS total_users,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) = 0 THEN 1 ELSE 0 END) AS ordered_same_day,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ordered_same_day,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 7 THEN 1 ELSE 0 END) AS ordered_within_7d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 7 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ordered_7d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 15 THEN 1 ELSE 0 END) AS ordered_within_15d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 15 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ordered_15d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 30 THEN 1 ELSE 0 END) AS ordered_within_30d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 30 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ordered_30d,
    SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 90 THEN 1 ELSE 0 END) AS ordered_within_90d,
    ROUND(100.0 * SUM(CASE WHEN DATEDIFF('day', first_account_visit, first_new_order_at) <= 90 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ordered_90d
FROM baseline_analysis;

In [ ]:
print("--- Step 7: Historical New Order Baseline vs Test Branches ---")
print("\nHistorical Baseline (6 months pre-test, account page visitors):")
print(step9_baseline.to_string(index=False))

# Extract baseline values
baseline_o = step9_baseline.iloc[0]

# Extract test branch values
control_o = step9_orders[step9_orders['TEST_BRANCH'] == 'Control'].iloc[0]
variant_o = step9_orders[step9_orders['TEST_BRANCH'] == 'Variant'].iloc[0]

print(f"\nComparison Table:")
comparison_o = pd.DataFrame({
    'Window': ['Same Day', '7 Days', '15 Days', '30 Days', '90 Days'],
    'Historical Baseline': [
        f"{baseline_o['PCT_ORDERED_SAME_DAY']:.2f}%",
        f"{baseline_o['PCT_ORDERED_7D']:.2f}%",
        f"{baseline_o['PCT_ORDERED_15D']:.2f}%",
        f"{baseline_o['PCT_ORDERED_30D']:.2f}%",
        f"{baseline_o['PCT_ORDERED_90D']:.2f}%"
    ],
    'Control': [
        f"{control_o['PCT_ORDERED_SAME_DAY']:.2f}%",
        f"{control_o['PCT_ORDERED_7D']:.2f}%",
        f"{control_o['PCT_ORDERED_15D']:.2f}%",
        f"{control_o['PCT_ORDERED_30D']:.2f}%",
        f"{control_o['PCT_ORDERED_90D']:.2f}%"
    ],
    'Variant': [
        f"{variant_o['PCT_ORDERED_SAME_DAY']:.2f}%",
        f"{variant_o['PCT_ORDERED_7D']:.2f}%",
        f"{variant_o['PCT_ORDERED_15D']:.2f}%",
        f"{variant_o['PCT_ORDERED_30D']:.2f}%",
        f"{variant_o['PCT_ORDERED_90D']:.2f}%"
    ]
})
print(comparison_o.to_string(index=False))

# Visualization
windows = ['Same Day', '7 Days', '15 Days', '30 Days', '90 Days']
baseline_vals = [baseline_o['PCT_ORDERED_SAME_DAY'], baseline_o['PCT_ORDERED_7D'], baseline_o['PCT_ORDERED_15D'], baseline_o['PCT_ORDERED_30D'], baseline_o['PCT_ORDERED_90D']]
control_vals = [control_o['PCT_ORDERED_SAME_DAY'], control_o['PCT_ORDERED_7D'], control_o['PCT_ORDERED_15D'], control_o['PCT_ORDERED_30D'], control_o['PCT_ORDERED_90D']]
variant_vals = [variant_o['PCT_ORDERED_SAME_DAY'], variant_o['PCT_ORDERED_7D'], variant_o['PCT_ORDERED_15D'], variant_o['PCT_ORDERED_30D'], variant_o['PCT_ORDERED_90D']]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(windows))
width = 0.25

bars1 = ax.bar(x - width, baseline_vals, width, label='Historical Baseline', color='#9E9E9E')
bars2 = ax.bar(x, control_vals, width, label='Control', color='#2196F3')
bars3 = ax.bar(x + width, variant_vals, width, label='Variant', color='#FF9800')

ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=8)
ax.bar_label(bars3, fmt='%.1f%%', padding=3, fontsize=8)

ax.set_ylabel('New Order Rate (%)')
ax.set_title('New Order Rate by Time Window: Historical Baseline vs Test Branches')
ax.set_xticks(x)
ax.set_xticklabels(windows)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8: Statistical Validation

Run chi-squared tests and calculate confidence intervals to validate whether differences between Control and Variant are statistically significant for both cancellation rates and new order rates.

In [ ]:
from scipy.stats import chi2_contingency, norm
import warnings
warnings.filterwarnings('ignore')

def run_proportion_test(name, control_n, control_events, variant_n, variant_events):
    """Run a two-proportion z-test and chi-squared test."""
    p_control = control_events / control_n if control_n > 0 else 0
    p_variant = variant_events / variant_n if variant_n > 0 else 0
    
    # Chi-squared test
    observed = np.array([[control_events, control_n - control_events],
                         [variant_events, variant_n - variant_events]])
    
    if observed.min() >= 0 and control_n > 0 and variant_n > 0:
        chi2, p_value, dof, expected = chi2_contingency(observed)
    else:
        chi2, p_value = 0, 1.0
    
    # Relative lift
    lift = ((p_variant - p_control) / p_control * 100) if p_control > 0 else 0
    
    # 95% confidence interval for the difference
    p_pooled = (control_events + variant_events) / (control_n + variant_n)
    
    #se = np.sqrt(p_pooled * (1 - p_pooled) * (1/control_n + 1/variant_n)) if (control_n > 0 and variant_n > 0) else 0
    
    se_unpooled = np.sqrt(
    (p_control * (1 - p_control) / control_n) +
    (p_variant * (1 - p_variant) / variant_n))
    
    ci_low  = (p_variant - p_control) - 1.96 * se_unpooled
    ci_high = (p_variant - p_control) + 1.96 * se_unpooled
    
    z_score = (p_variant - p_control) / se_unpooled if se_unpooled > 0 else 0
    
    ##ci_low = (p_variant - p_control) - 1.96 * se
    ##ci_high = (p_variant - p_control) + 1.96 * se
    
    significance = "YES" if p_value < 0.05 else "NO"
    
    return {
        'Metric': name,
        'Control Rate': f'{p_control*100:.2f}%',
        'Variant Rate': f'{p_variant*100:.2f}%',
        'Relative Lift': f'{lift:+.2f}%',
        'Chi2 Stat': f'{chi2:.4f}',
        'P-Value': f'{p_value:.4f}',
        'CI (95%)': f'[{ci_low*100:.2f}%, {ci_high*100:.2f}%]',
        'Significant (p<0.05)': significance
    }

# Get values from step 6 (cancellations)
control_cancel = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Control'].iloc[0]
variant_cancel = step7_cancellations[step7_cancellations['TEST_BRANCH'] == 'Variant'].iloc[0]

# Get values from step 7 (orders)
control_order = step9_orders[step9_orders['TEST_BRANCH'] == 'Control'].iloc[0]
variant_order = step9_orders[step9_orders['TEST_BRANCH'] == 'Variant'].iloc[0]

results = []

# Cancellation tests
for window, col in [('Same Day', 'CANCELLED_SAME_DAY'), ('7 Days', 'CANCELLED_WITHIN_7D'), 
                     ('15 Days', 'CANCELLED_WITHIN_15D'), ('30 Days', 'CANCELLED_WITHIN_30D'),
                    ('90 Days', 'CANCELLED_WITHIN_90D')]:
    results.append(run_proportion_test(
        f'Cancellation ({window})',
        int(control_cancel['TOTAL_USERS']), int(control_cancel[col]),
        int(variant_cancel['TOTAL_USERS']), int(variant_cancel[col])
    ))

# New order tests
for window, col in [('Same Day', 'ORDERED_SAME_DAY'), ('7 Days', 'ORDERED_WITHIN_7D'),
                     ('15 Days', 'ORDERED_WITHIN_15D'), ('30 Days', 'ORDERED_WITHIN_30D'),
                    ('90 Days', 'ORDERED_WITHIN_90D')]:
    results.append(run_proportion_test(
        f'New Order ({window})',
        int(control_order['TOTAL_USERS']), int(control_order[col]),
        int(variant_order['TOTAL_USERS']), int(variant_order[col])
    ))

results_df = pd.DataFrame(results)
print("--- Step 8: Statistical Validation Results ---")
print("\n" + results_df.to_string(index=False))

In [ ]:
# Visualization of p-values
fig, ax = plt.subplots(figsize=(12, 5))

metric_names = results_df['Metric'].tolist()
p_values = [float(x) for x in results_df['P-Value'].tolist()]

colors = ['#4CAF50' if p < 0.05 else '#F44336' for p in p_values]
bars = ax.barh(metric_names, p_values, color=colors)
ax.axvline(x=0.05, color='black', linestyle='--', linewidth=2, label='p=0.05 threshold')
ax.set_xlabel('P-Value')
ax.set_title('Statistical Significance: P-Values by Metric\n(Green = Significant, Red = Not Significant)')
ax.legend()
ax.set_xlim(0, max(max(p_values) * 1.1, 0.1))

for bar, p in zip(bars, p_values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{p:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 9: User Behavior Analysis

Analyze navigation behavior between branches: session duration, pages visited, time spent on account site, loyalty points redemption, and other engagement metrics.

In [ ]:
%%sql -r step11_behavior
-- Step 9: User behavior analysis by branch
WITH session_metrics AS (
    SELECT
        test_branch,
        session_id,
        canonical_id,
        COUNT(*) AS events_in_session,
        COUNT(DISTINCT page_location) AS distinct_pages,
        DATEDIFF('second', MIN(event_timestamp_utc), MAX(event_timestamp_utc)) AS session_duration_seconds,
        COUNT(CASE WHEN page_type = 'Account' THEN 1 END) AS account_page_events,
        DATEDIFF('second',
            MIN(CASE WHEN page_type = 'Account' THEN event_timestamp_utc END),
            MAX(CASE WHEN page_type = 'Account' THEN event_timestamp_utc END)
        ) AS time_on_account_seconds,
        COUNT(CASE WHEN page_type = 'Reward' THEN 1 END) AS reward_events,
        COUNT(CASE WHEN event_name = 'page_view' THEN 1 END) AS page_views,
        COUNT(CASE WHEN event_name LIKE '%click%' THEN 1 END) AS click_events,
        COUNT(CASE WHEN event_name = 'add_to_cart' THEN 1 END) AS add_to_cart_events,
        COUNT(CASE WHEN page_type = 'PDP' THEN 1 END) AS pdp_events
    FROM _test_events
    GROUP BY test_branch, session_id, canonical_id
)
SELECT
    test_branch,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(DISTINCT canonical_id) AS total_users,
    ROUND(MEDIAN(session_duration_seconds), 1) AS median_session_duration_sec,
    ROUND(MEDIAN(distinct_pages), 1) AS median_pages_per_session,
    ROUND(MEDIAN(events_in_session), 1) AS median_events_per_session,
    ROUND(MEDIAN(account_page_events), 1) AS median_account_page_events,
    ROUND(MEDIAN(COALESCE(time_on_account_seconds, 0)), 1) AS median_time_on_account_sec,
    SUM(CASE WHEN reward_events > 0 THEN 1 ELSE 0 END) AS sessions_with_rewards,
    ROUND(100.0 * SUM(CASE WHEN reward_events > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_sessions_with_rewards,
    ROUND(MEDIAN(page_views), 1) AS median_page_views,
    ROUND(MEDIAN(click_events), 1) AS median_clicks,
    SUM(CASE WHEN add_to_cart_events > 0 THEN 1 ELSE 0 END) AS sessions_with_add_to_cart,
    ROUND(100.0 * SUM(CASE WHEN add_to_cart_events > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_add_to_cart,
    SUM(CASE WHEN pdp_events > 0 THEN 1 ELSE 0 END) AS sessions_with_pdp,
    ROUND(100.0 * SUM(CASE WHEN pdp_events > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_pdp
FROM session_metrics
GROUP BY test_branch
ORDER BY test_branch;

In [ ]:
print("--- Step 9: User Behavior Analysis by Branch ---")
print(step11_behavior.T.to_string())

In [ ]:
# Behavior visualizations
fig, axes = plt.subplots(2, 4, figsize=(16, 10))

behavior_metrics = [
    ('MEDIAN_SESSION_DURATION_SEC', 'Median Session Duration (sec)'),
    ('MEDIAN_PAGES_PER_SESSION', 'Median Pages per Session'),
    ('MEDIAN_EVENTS_PER_SESSION', 'Median Events per Session'),
    ('MEDIAN_TIME_ON_ACCOUNT_SEC', 'Median Time on Account (sec)'),
    ('PCT_SESSIONS_WITH_REWARDS', '% Sessions with Rewards Activity'),
    ('MEDIAN_CLICKS', 'Median Clicks per Session'),
    ('PCT_ADD_TO_CART', '% Sessions with Add to Cart'),
    ('PCT_PDP', '% Sessions with PDP')
]

for idx, (col, title) in enumerate(behavior_metrics):
    ax = axes[idx // 4][idx % 4]
    vals = step11_behavior[['TEST_BRANCH', col]].copy()
    bars = ax.bar(vals['TEST_BRANCH'], vals[col], color=['#2196F3', '#FF9800'])
    ax.set_title(title)
    ax.bar_label(bars, fmt='%.1f', padding=3)

plt.suptitle('User Behavior Comparison: Control vs Variant', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Summary and Conclusion - TBD